# CN230 LCI — Semantic Search บน Render

Notebook นี้ใช้แนวคิดจากกิจกรรม W14 แต่เชื่อมกับข้อมูลจริงใน Render PostgreSQL ของโปรเจกต์ CN230 LCI โดยตรง

**Notebook นี้ไม่โหลดหรือเขียนข้อมูล LCI ซ้ำ** แต่สร้าง embeddings สำหรับแถวใน `lci_exchange` แล้วบันทึกลง `semantic_search_documents` ที่มีอยู่

## ก่อนเริ่ม

1. เปิด Colab → Secrets (ไอคอนกุญแจ) แล้วเพิ่ม `DATABASE_URL` เป็น Render **External Database URL** และเพิ่ม `GEMINI_API_KEY`
2. ใช้รหัสผ่าน DB และ Gemini key ที่หมุนเวียนใหม่แล้ว อย่าวาง secret ลงในเซลล์หรือ output
3. ตรวจว่า Render DB มี LCI ครบ และ Web Service มี `GEMINI_API_KEY` ใน Environment
4. รันเซลล์ตามลำดับ; เริ่มที่ 100 แถวก่อนเพื่อตรวจผล จากนั้นค่อยเลือกทำทั้งหมด

Embeddings ใช้ `gemini-embedding-001` 768 มิติ ให้ตรงกับ API ของเว็บ CN230 LCI


In [ ]:
%pip -q install "psycopg[binary]>=3.2,<4" "google-genai>=1,<4" "pgvector>=0.3,<1" pandas


In [ ]:
import json
import time
from math import sqrt
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

import psycopg
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata

GEMINI_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSIONS = 768

raw_database_url = (userdata.get("DATABASE_URL") or "").strip()
GEMINI_API_KEY = (userdata.get("GEMINI_API_KEY") or "").strip()
if not raw_database_url:
    raise RuntimeError("เพิ่ม DATABASE_URL ใน Colab Secrets ก่อนรัน")
if not GEMINI_API_KEY:
    raise RuntimeError("เพิ่ม GEMINI_API_KEY ใน Colab Secrets ก่อนรัน")

# Render requires TLS for external Postgres connections. Never print the URL or key.
parts = urlsplit(raw_database_url)
params = dict(parse_qsl(parts.query, keep_blank_values=True))
params["sslmode"] = "require"
DATABASE_URL = urlunsplit((parts.scheme, parts.netloc, parts.path, urlencode(params), parts.fragment))
ai = genai.Client(api_key=GEMINI_API_KEY)

conn = psycopg.connect(DATABASE_URL, connect_timeout=20)
info = conn.execute("SELECT current_database(), current_user").fetchone()
print(f"Connected to Render database: {info[0]} (user: {info[1]})")


## 1. เตรียมตารางเวกเตอร์

สร้างเฉพาะ extension, ตาราง และดัชนี Semantic Search แบบ `IF NOT EXISTS` ไม่ลบหรือนำเข้าตาราง LCI ใหม่


In [ ]:
conn.execute("CREATE EXTENSION IF NOT EXISTS vector;")
conn.execute("""
    CREATE TABLE IF NOT EXISTS semantic_search_documents (
        document_id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        entity_type text NOT NULL,
        entity_id bigint NOT NULL,
        title text NOT NULL,
        content text NOT NULL,
        source_code varchar(20),
        metadata jsonb NOT NULL DEFAULT '{}'::jsonb,
        embedding vector(768) NOT NULL,
        indexed_at timestamptz NOT NULL DEFAULT now(),
        UNIQUE (entity_type, entity_id)
    );
""")
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_semantic_search_documents_embedding
    ON semantic_search_documents USING hnsw (embedding vector_cosine_ops);
""")
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_semantic_search_documents_source
    ON semantic_search_documents (source_code);
""")
conn.commit()
print("Semantic Search schema is ready.")


## 2. ตรวจข้อมูล LCI ใน Render

ถ้ามี `exchange_id` ซ้ำ ให้หยุดก่อนสร้างเวกเตอร์ เพราะผลค้นหาอาจปนข้อมูลจากการ restore ซ้ำ


In [ ]:
total_rows, unique_ids = conn.execute(
    "SELECT COUNT(*), COUNT(DISTINCT exchange_id) FROM lci_exchange"
).fetchone()
indexed_rows = conn.execute(
    "SELECT COUNT(*) FROM semantic_search_documents WHERE entity_type = 'exchange'"
).fetchone()[0]
print(f"LCI Exchange rows: {total_rows:,}")
print(f"Unique exchange IDs: {unique_ids:,}")
print(f"Already indexed: {indexed_rows:,}")
if total_rows != unique_ids:
    raise RuntimeError("พบ exchange_id ซ้ำใน Render DB กรุณาแก้ข้อมูลก่อน index")

src_counts = conn.execute(
    "SELECT s.src_code, COUNT(*) FROM lci_exchange e "
    "JOIN source s ON s.source_id = e.source_id "
    "GROUP BY s.src_code ORDER BY s.src_code"
).fetchall()
display(pd.DataFrame(src_counts, columns=["SRC", "exchange_rows"]))


## 3. สร้าง embeddings แล้ว UPSERT เป็นชุด

คิวรีจะเลือกเฉพาะ Exchange ที่ยังไม่มี embedding หรือข้อมูลสำคัญเปลี่ยนไป การรันซ้ำจึงต่อจากรายการที่ยังค้างได้

เริ่มที่ 100 รายการก่อน ถ้าผลลัพธ์ถูกต้องให้เปลี่ยน `INDEX_LIMIT` เป็น `0` เพื่อทำทั้งหมด


In [ ]:
ROWS_SQL = """
SELECT e.exchange_id, s.src_code, s.dataset_name, s.agency,
       p.name, p.process_type, f.name, f.product_type, pf.direction,
       e.amount::text, u.symbol, e.period_year, e.period_month,
       e.validation_status
FROM lci_exchange e
JOIN process_flow pf ON pf.process_flow_id = e.process_flow_id
JOIN process p ON p.process_id = pf.process_id
JOIN flow f ON f.flow_id = pf.flow_id
JOIN unit u ON u.unit_id = e.unit_id
JOIN source s ON s.source_id = e.source_id
LEFT JOIN semantic_search_documents d
       ON d.entity_type = 'exchange' AND d.entity_id = e.exchange_id
WHERE d.document_id IS NULL
   OR d.metadata->>'validation_status' IS DISTINCT FROM e.validation_status
   OR d.metadata->>'amount' IS DISTINCT FROM e.amount::text
   OR d.metadata->>'unit' IS DISTINCT FROM u.symbol
   OR d.metadata->>'process_name' IS DISTINCT FROM p.name
   OR d.metadata->>'process_type' IS DISTINCT FROM p.process_type
   OR d.metadata->>'flow_name' IS DISTINCT FROM f.name
   OR d.metadata->>'product_type' IS DISTINCT FROM f.product_type
   OR d.metadata->>'direction' IS DISTINCT FROM pf.direction
   OR d.metadata->>'period_year' IS DISTINCT FROM e.period_year::text
   OR d.metadata->>'period_month' IS DISTINCT FROM e.period_month::text
   OR d.metadata->>'dataset_name' IS DISTINCT FROM s.dataset_name
ORDER BY e.exchange_id
"""

UPSERT_SQL = """
INSERT INTO semantic_search_documents
    (entity_type, entity_id, title, content, source_code, metadata, embedding, indexed_at)
VALUES ('exchange', %s, %s, %s, %s, %s::jsonb, %s::vector, now())
ON CONFLICT (entity_type, entity_id) DO UPDATE
SET title = EXCLUDED.title,
    content = EXCLUDED.content,
    source_code = EXCLUDED.source_code,
    metadata = EXCLUDED.metadata,
    embedding = EXCLUDED.embedding,
    indexed_at = now()
"""

def make_document(row):
    (exchange_id, src_code, dataset_name, agency, process_name, process_type,
     flow_name, product_type, direction, amount, unit, year, month, status) = row
    period = f"{year}-{month:02d}" if month else str(year)
    title = f"{src_code}: {flow_name} / {process_name} / {period}"
    content = (
        f"LCI exchange. Source {src_code}, dataset {dataset_name}, agency {agency}. "
        f"Process {process_name}, type {process_type}. Flow/product {flow_name}, "
        f"product type {product_type}, direction {direction}. Amount {amount} {unit}. "
        f"Period {period}. Validation status {status}. "
        f"รายการ LCI จากแหล่ง {src_code} ชุดข้อมูล {dataset_name}. "
        f"กระบวนการ {process_name} ประเภท {process_type}. ผลิตภัณฑ์หรือการไหล {flow_name}. "
        f"ปริมาณ {amount} {unit} ช่วงเวลา {period}."
    )
    metadata = {
        "dataset_name": dataset_name, "process_name": process_name,
        "process_type": process_type, "flow_name": flow_name,
        "product_type": product_type, "direction": direction,
        "amount": amount, "unit": unit, "period_year": year,
        "period_month": month, "validation_status": status,
    }
    return exchange_id, src_code, title, content, metadata

def index_pending(limit=100, batch_size=64, pause_seconds=0.2):
    rows = conn.execute(ROWS_SQL).fetchall()
    if limit:
        rows = rows[:limit]
    print(f"Rows needing embeddings: {len(rows):,}")
    for start in range(0, len(rows), batch_size):
        batch = [make_document(row) for row in rows[start:start + batch_size]]
        response = ai.models.embed_content(
            model=GEMINI_MODEL,
            contents=[doc[3] for doc in batch],
            config=types.EmbedContentConfig(
                task_type="RETRIEVAL_DOCUMENT",
                output_dimensionality=EMBEDDING_DIMENSIONS,
            ),
        )
        if not response.embeddings or len(response.embeddings) != len(batch):
            raise RuntimeError("Gemini returned an unexpected number of embeddings")

        values = []
        for doc, embedded in zip(batch, response.embeddings):
            vector = embedded.values
            if vector is None or len(vector) != EMBEDDING_DIMENSIONS:
                raise RuntimeError("Embedding dimension does not match vector(768)")
            norm = sqrt(sum(value * value for value in vector))
            if not norm:
                raise RuntimeError("Gemini returned a zero vector")
            vector_literal = "[" + ",".join(str(value / norm) for value in vector) + "]"
            values.append((doc[0], doc[2], doc[3], doc[1], json.dumps(doc[4], ensure_ascii=False), vector_literal))

        with conn.cursor() as cur:
            cur.executemany(UPSERT_SQL, values)
        conn.commit()
        done = min(start + len(batch), len(rows))
        print(f"Indexed {done:,}/{len(rows):,}")
        if pause_seconds and done < len(rows):
            time.sleep(pause_seconds)

INDEX_LIMIT = 100  # ตั้งเป็น 0 หลังตรวจชุดทดลอง เพื่อ index รายการที่เหลือทั้งหมด


In [ ]:
index_pending(limit=INDEX_LIMIT)


## 4. ตรวจจำนวนที่ index แล้ว


In [ ]:
exchange_count = conn.execute(
    "SELECT COUNT(*) FROM lci_exchange"
).fetchone()[0]
indexed_count = conn.execute(
    "SELECT COUNT(*) FROM semantic_search_documents WHERE entity_type = 'exchange'"
).fetchone()[0]
pending_count = conn.execute(ROWS_SQL).fetchall()
print(f"Exchange in Render DB: {exchange_count:,}")
print(f"Exchange indexed: {indexed_count:,}")
print(f"Still missing or changed: {len(pending_count):,}")


## 5. ค้นหาแบบ Semantic Search

ใช้ `RETRIEVAL_QUERY` และโมเดล/ขนาดเวกเตอร์เดียวกับที่ใช้จัดทำเอกสาร


In [ ]:
def search_lci(query, limit=10, source_code=None):
    response = ai.models.embed_content(
        model=GEMINI_MODEL,
        contents=query,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=EMBEDDING_DIMENSIONS,
        ),
    )
    vector = response.embeddings[0].values
    norm = sqrt(sum(value * value for value in vector))
    query_vector = "[" + ",".join(str(value / norm) for value in vector) + "]"
    return conn.execute(
        """
        SELECT entity_type, entity_id, title, content, source_code, metadata,
               1 - (embedding <=> %s::vector) AS similarity
        FROM semantic_search_documents
        WHERE entity_type = 'exchange'
          AND (%s IS NULL OR source_code = %s)
        ORDER BY embedding <=> %s::vector
        LIMIT %s
        """,
        (query_vector, source_code, source_code, query_vector, limit),
    ).fetchall()

QUESTION = "การผลิตไฟฟ้าจากถ่านหินมีข้อมูลอะไรบ้าง"
hits = search_lci(QUESTION, limit=10)
display(pd.DataFrame([
    {
        "title": row[2], "source": row[4],
        "similarity": round(float(row[6]), 4), "evidence": row[3],
    }
    for row in hits
]))


## 6. ตอบโดยอ้างอิงหลักฐาน (เลือกใช้)

เซลล์นี้เรียก Gemini เพิ่มอีกหนึ่งครั้ง ใช้เฉพาะข้อความที่ค้นพบและให้ระบุ SRC ที่อ้างอิง


In [ ]:
RUN_GROUNDED_ANSWER = False  # เปลี่ยนเป็น True เมื่อพร้อมทดลองการสร้างคำตอบ

if RUN_GROUNDED_ANSWER:
    evidence = "\n".join(
        f"[{i}] SRC={row[4]}, title={row[2]}, text={row[3]}"
        for i, row in enumerate(hits, start=1)
    )
    prompt = (
        "ตอบเป็นภาษาไทยโดยใช้เฉพาะหลักฐานต่อไปนี้ ห้ามแต่งข้อมูลเพิ่ม "
        "ถ้าหลักฐานไม่พอให้ตอบว่า ไม่พบข้อมูลที่เกี่ยวข้อง "
        "และอ้างอิงรหัส SRC ที่ใช้\n\n"
        f"คำถาม: {QUESTION}\n\nหลักฐาน:\n{evidence}"
    )
    answer = ai.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    print(answer.text or "ไม่พบข้อมูลที่เกี่ยวข้อง")


## หมายเหตุ

- ถ้า `INDEX_LIMIT = 100` แสดงผลถูกต้อง ให้เปลี่ยนเป็น `0` แล้วรันเซลล์สร้าง embeddings ซ้ำเพื่อทำรายการที่เหลือ
- หาก Colab หลุดกลางทาง ให้ reconnect แล้วรันตั้งแต่เชื่อมต่อ DB; UPSERT จะข้ามรายการเดิมที่ไม่เปลี่ยน
- เมื่อจบให้ตรวจว่าจำนวน indexed ตรงกับ Exchange ใน Render และ `Still missing or changed` เป็น 0
- ปิดการเชื่อมต่อเมื่อเสร็จด้วย `conn.close()`
- ถ้าต้องให้เว็บ Render ค้นหาได้ ให้ตั้ง `GEMINI_API_KEY` ใน Environment ของ Web Service ด้วย
